<a href="https://colab.research.google.com/github/Hassanmufezshaikh/AI-Agents/blob/main/SequentialAgentArictecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-adk google-genai

  Using cached google_genai-2.4.0-py3-none-any.whl.metadata (52 kB)


In [ ]:
import google.adk
print(google.adk.__version__)

2.0.0


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print(" Tunnel Components imported successfully")



 Tunnel Components imported successfully


In [ ]:
import os
from google.colab import userdata
userdata.get('gemeni')

try:
  GOOGLE_API_KEY = userdata.get('gemeni')
  os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
  print("Gemini API Key Setup Complete")
except Exception as e :
  print("Authencation Error: Please add 'GEMENI_API_KEY' to your kaggale secrets, Details : {e}")

Gemini API Key Setup Complete


In [ ]:
from google.adk.agents import (
    Agent,
    SequentialAgent,
    ParallelAgent,
    LoopAgent
)

from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types

print("ADK components imported successfully.")

ADK components imported successfully.


In [ ]:
import google.adk.tools as tools

print(dir(tools))

['APIHubToolset', 'AgentTool', 'AgentTool', 'Any', 'ApiRegistry', 'AuthToolArguments', 'BaseTool', 'DiscoveryEngineSearchTool', 'ExampleTool', 'FunctionTool', 'FunctionTool', 'LongRunningFunctionTool', 'MCPToolset', 'McpToolset', 'SearchResultMode', 'TYPE_CHECKING', 'ToolContext', 'TransferToAgentTool', 'VertexAiSearchTool', '_LAZY_MAPPING', '__all__', '__builtins__', '__cached__', '__dir__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_automatic_function_calling_util', '_forwarding_artifact_service', '_function_parameter_parse_util', '_function_tool_declarations', 'agent_tool', 'base_tool', 'base_toolset', 'computer_use', 'enterprise_web_search', 'exit_loop', 'function_tool', 'get_user_choice', 'google_maps_grounding', 'google_search', 'google_search', 'google_search_tool', 'importlib', 'load_artifacts', 'load_memory', 'logging', 'preload_memory', 'set_model_response_tool', 'sys', 'tool_configs', 'tool_confirmation', 'tool_co

In [ ]:
from google.genai import types

retry_config=types.HttpRetryOptions(
attempts=5, # Maximum retry attempts
exp_base=7, # Delay multiplier
initial_delay=1, # Initial delay before first retry (in seconds)
http_status_codes=[429, 500, 503, 504]
)

In [ ]:
# Research Agent: Its job is to use the google_search tool and present findings.

research_agent = Agent(
    name="ResearchAgent",

    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),

    instruction="""
    You are a specialized research agent.
    Your only job is to use the Google Search tool
    to find 2-3 pieces of relevant information
    on the given topic and present the findings with citations.
    """,

    tools=[google_search],

    # The result of this agent will be stored in session state
    output_key="research_findings",
)

print("research_agent created.")

research_agent created.


In [ ]:
#Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
name="SummarizerAgent",
model=Gemini(
model="gemini-2.5-flash-lite",
retry_options=retry_config
),
# The instruction is modified to request a bulleted list for a clear output format.
instruction="""Read the provided research findings: {research_findings},
create a concise summary as a bulleted list with 3-5 key points.
""",
output_key="final_summary"
)
print(" summarizer_agent created.")

 summarizer_agent created.


In [ ]:
# # @title Default title text
# #Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
# root_agent = Agent (
# name="ResearchCoordinator",
# model=Gemini(
# model="gemini-2.5-flash-lite",
# retry_options=retry_config
# ),
# #This instruction tells the root agent HOW to use its tools (which are the other agents).
# instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
# 1. First, you MUST call the ResearchAgent tool to find relevant information on the topic provided by the user.
# 2. Next, after receiving the research findings, you MUST call the 'SummarizerAgent tool to create a concise summary.
# 3. Finally, present the final summary clearly to the user as your response.""",
# )

# #We wrap the sub-agents in AgentTool to make them callable tools for the root agent.
# tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
# print(" root_agent created.")

 root_agent created.


In [ ]:
# runner = InMemoryRunner(agent=root_agent)
# print("InMemoryRunner created.")
# response =  await runner.run_debug(
#     "What are the Latest Advancements in quantum computing and what they do mean for AI?"
# )

InMemoryRunner created.
ResearchCoordinator > Researching the latest advancements in quantum computing and their implications for AI.
**ResearchAgent**:
Please provide me with the topic you'd like me to research. For example, "latest advancements in quantum computing and their implications for artificial intelligence."
I'm ready to find the information you need.The user has already provided the topic. Please proceed to call the ResearchAgent tool with the user's query.
**ResearchAgent**:
What are the latest advancements in quantum computing and what they mean for AI?
The latest advancements in quantum computing show significant progress in several key areas. These include:

*   **Increased Qubit Stability and Coherence Times**: Researchers are developing more robust qubits that can maintain their quantum state for longer periods. This is crucial for performing complex calculations without errors. Recent breakthroughs have seen coherence times extended into the milliseconds and even sec

In [ ]:
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Create a blog outline for the given topic with:
1. A catchy headline
2. An introduction hook
3. 3-5 main sections with 2-3 bullet points for each
4. A concluding thought""",
    output_key="blog_outline", #The result of this agent will be stored in the session state with this key.
)
print(" outline_agent created.")

 outline_agent created.


In [ ]:
# Writer Agent: Writes the full blog post based on the outline from the previous agent.
writer_agent = Agent (
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    #The {blog_outline} placeholder automatically injects the state value from the previous agent's output.
    instruction="""Following this outline strictly: {blog_outline}
Write a brief, 200 to 300-word blog post with an engaging and informative the.""",
    output_key="blog_draft", # The result of this agent will be stored with this key.
)
print(" writer_agent created.")

 writer_agent created.


In [ ]:
# Editor Agent: Edites and polishes the draft from the writer agent.
editor_agent = Agent (
    name="EditorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    #The Agents recives the Blog draft from the writer agnets.
    instruction="""Edit these draft: {blog_draft}
your task is to polish the text by fixing ang grammatical errors, improving the flow sentence structure, and enhnacing overall capacity.""",
    output_key="final_bllog", # The result of this agent will be stored with this key.
)
print(" Editor_agent created.")

 Editor_agent created.


In [ ]:
#Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = SequentialAgent (
name="BlogPipeLine",
sub_agents=[outline_agent, writer_agent, editor_agent]
)
print(" Sequential_agent created.")

 Sequential_agent created.


/tmp/ipykernel_23599/2562197351.py:2: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  root_agent = SequentialAgent (


In [ ]:
runner = InMemoryRunner(agent=root_agent)
print("InMemoryRunner created.")
response =  await runner.run_debug(
    "Write a blog post about the beneifts of multi-agents systems for software development"
)

InMemoryRunner created.
OutlineAgent > ## OutlineAgent: Blog Post Outline Request Received

**Topic:** Benefits of Multi-Agent Systems for Software Development

**Objective:** Create a blog post outline with a catchy headline, introduction hook, 3-5 main sections (each with 2-3 bullet points), and a concluding thought.

---

### Blog Post Outline:

**Catchy Headline:** **Unlock Agility & Resilience: How Multi-Agent Systems are Revolutionizing Software Development**

**Introduction Hook:** Imagine a software system that's not a monolithic giant, but a collaborative team of intelligent, independent agents, each with its own expertise and responsibilities. This is the promise of Multi-Agent Systems (MAS), a paradigm shift that's empowering developers to build more robust, adaptable, and efficient software than ever before.

**Main Sections:**

**1. Enhanced Modularity and Maintainability:**
    *   Break down complex problems into smaller, manageable agent-based components, leading to cle